# Nonstationary GP

入力位置によって latent function の smoothness が変わる場合に Gibbs kernel を使います。


## モデル構築


In [ ]:
from botorch.fit import fit_gpytorch_mll
import torch
from robotorchan.models import NonstationarySingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 16).unsqueeze(-1)
Y = torch.where(X < 0.5, torch.sin(2 * torch.pi * X), torch.sin(8 * torch.pi * X))
model = NonstationarySingleTaskGP(X, Y)
fit_gpytorch_mll(model.make_mll())


## 局所 lengthscale


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
print(model.local_lengthscale(test_X))
print(model.posterior(test_X).mean.squeeze(-1))


## Nonstationary candidate proposal

非定常カーネルによる事後平均から候補を選び、元の目的関数で評価します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.where(
    candidate < 0.5,
    torch.sin(2 * torch.pi * candidate),
    torch.sin(8 * torch.pi * candidate),
)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)


## Mixed / multi-task / Kroneckerの候補提案

単一出力・long-format multi-task・block-design Kroneckerの4モデルを追加します。有限候補プールの事後平均を比較し、カテゴリとtaskの構造を保持して目的関数を評価します。これは獲得関数の連続最適化ではありません。


In [ ]:
from robotorchan.models import (
    MixedNonstationaryMultiTaskGP,
    MixedNonstationarySingleTaskGP,
    NonstationaryKroneckerMultiTaskGP,
    NonstationaryMultiTaskGP,
)

def nonstationary_target(x):
    return torch.where(
        x < 0.5, torch.sin(2 * torch.pi * x),
        torch.sin(8 * torch.pi * x),
    )


design_x = torch.linspace(0.05, 0.95, 12).unsqueeze(-1)
category = torch.arange(len(design_x)).remainder(2).to(design_x).unsqueeze(-1)
mixed_design = torch.cat([design_x, category], dim=-1)
base_y = nonstationary_target(design_x)
mixed_y = base_y + 0.1 * category
task_y = base_y + 0.2
mixed_task_y = mixed_y + 0.2

long_X = torch.cat(
    [
        torch.cat([design_x, torch.zeros(len(design_x), 1)], dim=-1),
        torch.cat([design_x, torch.ones(len(design_x), 1)], dim=-1),
    ],
    dim=0,
)
long_Y = torch.cat([base_y, task_y], dim=0)
mixed_long_X = torch.cat(
    [
        torch.cat([mixed_design, torch.zeros(len(design_x), 1)], dim=-1),
        torch.cat([mixed_design, torch.ones(len(design_x), 1)], dim=-1),
    ],
    dim=0,
)
mixed_long_Y = torch.cat([mixed_y, mixed_task_y], dim=0)
block_Y = torch.cat([base_y, task_y], dim=-1)

nonstationary_models = [
    (
        "Mixed single-task",
        MixedNonstationarySingleTaskGP(
            mixed_design, mixed_y, cat_dims=[1]
        ),
    ),
    (
        "Multi-task",
        NonstationaryMultiTaskGP(
            long_X, long_Y, task_feature=1
        ),
    ),
    (
        "Mixed multi-task",
        MixedNonstationaryMultiTaskGP(
            mixed_long_X, mixed_long_Y, task_feature=2, cat_dims=[1]
        ),
    ),
    (
        "Kronecker multi-task",
        NonstationaryKroneckerMultiTaskGP(design_x, block_Y),
    ),
]
for name, surrogate in nonstationary_models:
    fit_gpytorch_mll(surrogate.make_mll())
    surrogate.eval()
    print(name, "trained")


In [ ]:
candidate_x = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
candidate_category = torch.cat(
    [
        torch.cat([candidate_x, torch.zeros(len(candidate_x), 1)], dim=-1),
        torch.cat([candidate_x, torch.ones(len(candidate_x), 1)], dim=-1),
    ],
    dim=0,
)
for name, surrogate in nonstationary_models:
    if name.startswith("Mixed"):
        pool = candidate_category
    else:
        pool = candidate_x
    with torch.no_grad():
        if name.endswith("multi-task") and name != "Kronecker multi-task":
            posterior = surrogate.posterior(pool, output_indices=[1])
        else:
            posterior = surrogate.posterior(pool)
        scores = posterior.mean
    if name == "Kronecker multi-task":
        scores = scores[..., 1]
    else:
        scores = scores.squeeze(-1)
    selected = scores.argmax().item()
    candidate = pool[selected:selected + 1]
    observed = nonstationary_target(candidate[:, :1])
    if name.startswith("Mixed"):
        observed = observed + 0.1 * candidate[:, 1:2]
        assert candidate[0, 1].item() in (0.0, 1.0)
    if "multi-task" in name:
        observed = observed + 0.2
    assert candidate.shape == (1, pool.shape[-1])
    assert torch.isfinite(observed).all()
    print(name, "candidate:", candidate, "objective:", observed)
